In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("raw_data/naukri_data_science_jobs_india.csv")

In [4]:
# Standardize column names
df.columns = df.columns.str.lower().str.strip().str.replace(" ", "_")

In [5]:
df.head(2)

,company,location,education,skills/description,job_type,salary,job_experience,job_role
0,UPL,"Bangalore/Bengaluru, Mumbai (All Areas)",Post-graduate,"python, MLT, statistical modeling, machine lea...",contract,939957.0,5-6,Senior Data Scientist
1,Walmart,Bangalore/Bengaluru,Post-graduate,"Data Science, Machine learning, Python, Azure,...",contract,2060767.0,1-7,Senior Data Scientist


In [6]:
df.isnull().sum()

company                  0
location                 0
education             1676
skills/description       0
job_type              1280
salary                 976
job_experience           0
job_role                 0
dtype: int64

In [7]:
#handling missing value
#job type
df['job_type'].value_counts(dropna=False)

job_type
contract     5366
full-time    5354
NaN          1280
Name: count, dtype: int64

In [8]:
df['job_type'] = df['job_type'].fillna(df['job_type'].mode()[0])

In [9]:
#salary
df['salary'].isnull().sum()

np.int64(976)

In [10]:
#convert to numeric
df['salary'] = pd.to_numeric(df['salary'], errors='coerce')

In [11]:
df['salary'] = df['salary'].fillna(df['salary'].median())

In [12]:
#education
df['education'].value_counts(dropna=False)

education
UG                1757
Post-graduate     1753
ug                1726
PG                1720
Under-graduate    1685
pg                1683
NaN               1676
Name: count, dtype: int64

In [13]:
df['education'] = df['education'].str.lower()

df['education'] = df['education'].replace({
    'ug': 'undergraduate',
    'under-graduate': 'undergraduate',
    'pg': 'postgraduate',
    'post-graduate': 'postgraduate'
})

df['education'] = df['education'].fillna('unknown')

In [14]:
#remove duplicates
df.drop_duplicates(inplace=True)

In [15]:
#experience feature engineering
df['min_experience'] = df['job_experience'].str.extract(r'(\d+)').astype(float)
df['max_experience'] = df['job_experience'].str.extract(r'-(\d+)').astype(float)

df['min_experience'] = df['min_experience'].fillna(0)
df['max_experience'] = df['max_experience'].fillna(df['min_experience'])

In [16]:
def experience_level(exp):
    if exp <= 2:
        return 'Entry'
    elif exp <= 5:
        return 'Mid'
    elif exp <= 10:
        return 'Senior'
    else:
        return 'Expert'

df['experience_level'] = df['max_experience'].apply(experience_level)

In [17]:
#skill feature engineering
import re
df.rename(columns={'skills/description': 'skills'}, inplace=True)
df['skills'] = df['skills'].str.lower()
df['skills'] = df['skills'].apply(lambda x: re.sub(r'[^a-zA-Z0-9,#+.\s]', ' ', x))
df['skills'] = df['skills'].apply(lambda x: re.sub(r'\s+', ' ', x).strip())

In [18]:
df['skills_list'] = df['skills'].apply(
    lambda x: re.split(r',|/|\||;', x)
)
df['skills_list'] = df['skills_list'].apply(
    lambda skills: [skill.strip() for skill in skills if skill.strip() != '']
)

In [19]:
df['skill_count'] = df['skills_list'].apply(len)
df['top_skill'] = df['skills_list'].apply(lambda x: x[0] if len(x) > 0 else 'unknown')

In [20]:
df.drop(columns=['skills_list'], inplace=True)

In [21]:
#salary
df["salary"] = pd.to_numeric(df["salary"], errors="coerce")
df.loc[df["salary"] <= 0, "salary"] = np.nan
salary_median = df["salary"].median()
df["salary"] = df["salary"].fillna(salary_median)

In [22]:
df["salary_lpa"] = df["salary"] / 100000
df["salary_lpa"] = df["salary_lpa"].round(2)

In [23]:
#location
df['location'] = df['location'].str.lower().str.strip()
#replace bracket
df['location'] = df['location'].str.replace(r'\(.*?\)', '', regex=True)
df['location'] = df['location'].str.replace('/', ',')
df['location'] = df['location'].str.replace('&', ',')

In [24]:
location_map = {
    'bangalore': 'bengaluru',
    'bengaluru': 'bengaluru',
    'delhi ncr': 'delhi',
    'new delhi': 'delhi',
    'delhi': 'delhi',
    'gurgaon': 'gurugram',
    'hyderabad secunderabad': 'hyderabad',
    'secunderabad': 'hyderabad',
    'bombay': 'mumbai',
    'calcutta': 'kolkata',
    'any location': 'remote',
    'work from home': 'remote'
}
def clean_location(loc):
    for key, val in location_map.items():
        loc = loc.replace(key, val)
    return loc

df['location'] = df['location'].apply(clean_location)

In [25]:
#location type
df['location_type'] = df['location'].apply(
    lambda x: 'Remote' if 'remote' in x else 'Onsite'
)

In [26]:
def get_first_location(loc):
    parts = [i.strip() for i in loc.split(',')]
    for p in parts:
        if p != '':
            return p
    return 'unknown'

df['primary_city'] = df['location'].apply(get_first_location)

In [27]:
df.loc[df['location_type'] == 'Remote', 'primary_city'] = 'remote'

In [28]:
df['job_type'] = df['job_type'].astype('category')
df['education'] = df['education'].astype('category')
df['experience_level'] = df['experience_level'].astype('category')

In [29]:
df.to_csv("cleaned_data/cleaned_naukri_dataset.csv", index=False)